In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import math



In [ ]:
train_df = pd.read_csv("Datast/train.csv")
test_df = pd.read_csv("Datast/test.csv")
building_df = pd.read_csv("Datast/building_metadata.csv")
weather_train_df = pd.read_csv("Datast/weather_train.csv")
weather_test_df = pd.read_csv("Datast/weather_test.csv")

In [ ]:
print("train_df_shape ",train_df.shape,'\n',train_df.isna().sum())
print("building_df_shape",building_df.shape,'\n',building_df.isna().sum())
print("weather_train_df_shape",weather_train_df.shape,'\n',weather_train_df.isna().sum())

In [ ]:
wet_cop=weather_train_df.copy()
buil_cop=building_df.copy()

In [ ]:
# wet_cop["timestamp"] = pd.to_datetime(wet_cop["timestamp"])
# wet_cop["hour"] = wet_cop["timestamp"].dt.hour
# wet_cop["day"] = wet_cop["timestamp"].dt.day
# wet_cop["month"] = wet_cop["timestamp"].dt.month
# wet_cop["weekend"] = (wet_cop["timestamp"].dt.weekday >= 5).astype(int)
# # wet_cop = wet_cop.set_index("timestamp")
    

In [ ]:
# print(wet_cop.info())          # Column types + missing counts
# print(wet_cop.describe())      # Numeric stats
# print(wet_cop.nunique())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

numeric_cols = wet_cop.select_dtypes(include=['float64', 'int64']).columns
corr_df = wet_cop[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(
    corr_df,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    mask=np.triu(np.ones_like(corr_df))  # Hide upper triangle for clarity
)
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
def saturation_vapor_pressure(temperature):
    return 6.112 * math.exp((17.67 * temperature) / (temperature + 243.5))

# Function to calculate relative humidity
def relative_humidity(air_temp, dew_temp):
    E_air = saturation_vapor_pressure(air_temp)
    E_dew = saturation_vapor_pressure(dew_temp)
    RH = (E_dew / E_air) * 100
    return RH

# Apply the function to each row in the dataframe to calculate relative humidity
wet_cop['relative_humidity'] = wet_cop.apply(lambda row: relative_humidity(row['air_temperature'], row['dew_temperature']), axis=1)


In [ ]:
# Simplified heat index (Rothfusz regression approximation)
wet_cop['heat_index'] = (
    0.5 * (wet_cop['air_temperature'] + 61.0 + 
    (wet_cop['air_temperature'] - 68.0) * 1.2 + 
    wet_cop['dew_temperature'] * 0.094
)
)

# Wind chill (for temps < 10°C and wind > 4.8 km/h)
mask = (wet_cop['air_temperature'] < 10) & (wet_cop['wind_speed'] > 1.34)  # 1.34 m/s ≈ 4.8 km/h
wet_cop['wind_chill'] = (
    13.12 + 0.6215 * wet_cop['air_temperature'] - 
    11.37 * (wet_cop['wind_speed'] ** 0.16) + 
    0.3965 * wet_cop['air_temperature'] * (wet_cop['wind_speed'] ** 0.16)
)
wet_cop['wind_chill'] = wet_cop['wind_chill'].where(mask, wet_cop['air_temperature'])

wet_cop['feels_like'] = np.where(
    wet_cop['air_temperature'] >= 27,  # Hot threshold
    wet_cop['heat_index'],
    np.where(
        wet_cop['air_temperature'] <= 10,  # Cold threshold
        wet_cop['wind_chill'],
        wet_cop['air_temperature']  # Default
    )
)

In [ ]:
wet_cop.columns

In [ ]:
wet_cop.hist(bins=50,figsize=(20,20))

In [ ]:

wet_cop['precip_depth_1_hr'] = (
    wet_cop['precip_depth_1_hr']
    .replace(-1.0, np.nan)
    .interpolate(method='linear', limit=6)  # 6-hour gap limit
    .fillna(0)  # Fill remaining gaps with 0
)

In [ ]:
bins = [-0.1, 0.1, 5.0, 15.0, float('inf')]  # -0.1 to catch near-zero values  
labels = ['No Rain', 'Light Rain', 'Moderate Rain', 'Heavy Rain']  

wet_cop['precip_1h_category'] = pd.cut(
    wet_cop['precip_depth_1_hr'],  
    bins=bins,  
    labels=labels  
)

In [ ]:
print(wet_cop['precip_1h_category'].value_counts(normalize=True) * 100)

In [ ]:
wet_cop['is_light_rain'] = (wet_cop['precip_depth_1_hr'] > 0.1).astype(int)  
# wet_cop['is_moderate_rain'] = (wet_cop['precip_depth_1_hr'] >= 5 and wet_cop['precip_depth_1_hr'] <15).astype(int)  
wet_cop['is_moderate_rain'] = ((wet_cop['precip_depth_1_hr'] > 0.1) & (wet_cop['precip_depth_1_hr'] < 15.0)).astype(int)

wet_cop['is_heavy_rain'] = (wet_cop['precip_depth_1_hr'] >= 15.0).astype(int)  

In [ ]:
Q1 = wet_cop['feels_like'].quantile(0.25)
Q3 = wet_cop['feels_like'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Flag outliers
wet_cop['feels_like_outlier'] = (
    (wet_cop['feels_like'] < lower_bound) | 
    (wet_cop['feels_like'] > upper_bound)
)

print(f"Found {wet_cop['feels_like_outlier'].sum()} outliers.")
print(f"Bounds: [{lower_bound:.1f}°C, {upper_bound:.1f}°C]")

In [ ]:
T_min = -10  # Below this, heating load is maxed
T_max = 35   # Above this, cooling load is maxed

wet_cop['feels_like_capped'] = wet_cop['feels_like'].clip(T_min, T_max)

In [ ]:
sns.boxplot(wet_cop['feels_like_capped'])

In [ ]:
rain_types = ['is_light_rain', 'is_moderate_rain', 'is_heavy_rain']
for rain_type in rain_types:
    subset = wet_cop[wet_cop[rain_type] == 1]
    outlier_pct = subset['feels_like_outlier'].mean() * 100
    print(f"{rain_type}: {outlier_pct:.1f}% outliers")

In [ ]:
# wet_cop['precip_6h_mean'] = (
#     wet_cop.groupby('site_id')['precip_depth_1_hr']
#     .transform(lambda x: x.rolling(6, min_periods=1).mean())
# )
wet_cop.columns.to_list()

In [ ]:
final_feat=['site_id','timestamp','air_temperature','sea_level_pressure',
 'wind_direction',
 'wind_speed',
 'relative_humidity',
 'is_light_rain',
 'is_moderate_rain',
 'is_heavy_rain',
 'feels_like_capped']

In [ ]:
final_weat_df = wet_cop[final_feat].copy()

In [ ]:
final_weat_df.shape

In [ ]:
missing_percentage = final_weat_df.isnull().sum() / len(final_weat_df) * 100
print(missing_percentage.sort_values(ascending=False))
# cp_main.hist(bins=50,figsize=(20,18))

In [ ]:
final_weat_df.dropna(subset=['sea_level_pressure','wind_direction','wind_speed','relative_humidity','feels_like_capped',"air_temperature"],inplace=True)

In [ ]:
final_weat_df

In [ ]:
final_weat_df.hist(bins=100,figsize=(20,20))

In [ ]:
train_merged = train_df.merge(buil_cop, on="building_id", how="left")
main_merged = train_merged.merge(final_weat_df, on=["site_id", "timestamp"], how="left",validate='many_to_one')
main_merged.shape

In [ ]:
main_merged.loc[(main_merged["site_id"] == 0) & (main_merged["meter"] == 0), "meter_reading"] *= 0.293071

In [ ]:
main_merged["meter_reading"] = np.log1p(main_merged["meter_reading"])


In [ ]:
drop_0readings= list(main_merged[main_merged['meter_reading']==0.0].index)
main_merged.drop(drop_0readings, axis=0, inplace=True)

In [ ]:
xdd=main_merged[main_merged['building_id']==45]

In [ ]:
main_merged['primary_use'] = main_merged['primary_use'].astype('category')
main_merged['meter'] = main_merged['meter'].astype('category')

In [ ]:
xdd.columns

In [ ]:
main_merged.iloc[1002:1010]

In [ ]:
import numpy as np
import pandas as pd

# Assuming main_merged is your weather dataframe
main_merged["timestamp"] = pd.to_datetime(main_merged["timestamp"])

# Extract basic time features
main_merged["hour"] = main_merged["timestamp"].dt.hour
main_merged["day_of_week"] = main_merged["timestamp"].dt.weekday  # Monday=0, Sunday=6
main_merged["month"] = main_merged["timestamp"].dt.month
main_merged["day_of_year"] = main_merged["timestamp"].dt.dayofyear
main_merged["is_weekend"] = (main_merged["day_of_week"] >= 5).astype(int)

# Cyclical encoding for hour
main_merged["hour_sin"] = np.sin(2 * np.pi * main_merged["hour"] / 24)
main_merged["hour_cos"] = np.cos(2 * np.pi * main_merged["hour"] / 24)

# Cyclical encoding for day of the week
main_merged["day_sin"] = np.sin(2 * np.pi * main_merged["day_of_week"] / 7)
main_merged["day_cos"] = np.cos(2 * np.pi * main_merged["day_of_week"] / 7)

# Cyclical encoding for month
main_merged["month_sin"] = np.sin(2 * np.pi * main_merged["month"] / 12)
main_merged["month_cos"] = np.cos(2 * np.pi * main_merged["month"] / 12)

# Season encoding (0: Winter, 1: Spring, 2: Summer, 3: Fall)
main_merged["season"] = main_merged["month"].map(lambda m: 0 if m in [12, 1, 2] else
                                         1 if m in [3, 4, 5] else
                                         2 if m in [6, 7, 8] else 3)

# Drop original timestamp column if no longer needed
main_merged.drop(columns=["timestamp"], inplace=True)

# Display updated dataframe
main_merged.head()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

numeric_cols = main_merged.select_dtypes(include=['float64', 'int64']).columns
corr_df = main_merged[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(
    corr_df,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    mask=np.triu(np.ones_like(corr_df))  # Hide upper triangle for clarity
)
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
# features = ["meter", "hour", "day", "month", "weekend",'sea_level_pressure']
features=['building_id', 'meter','site_id', 'primary_use',
       'square_feet', 'year_built',
       'sea_level_pressure', 'wind_direction', 'wind_speed',
       'relative_humidity', 'is_light_rain', 'is_moderate_rain',
       'is_heavy_rain', 'feels_like_capped', 'hour', 'day_of_week', 'month',
       'day_of_year', 'is_weekend', 'hour_sin', 'hour_cos', 'day_sin',
       'day_cos', 'month_sin', 'month_cos', 'season']
target = "meter_reading"

X = main_merged[features]
y = main_merged[target]

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "n_estimators": 1000,
    "verbose": -1
}

# Convert to LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Train the model
# model = lgb.train(params, train_data, valid_sets=[val_data], early_stopping_rounds=50, verbose_eval=100)
model = lgb.train(
    params,
    train_data,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)]
)


In [ ]:
from sklearn.metrics import r2_score
y_train_pred = model.predict(X_train)

# Calculate R² Score
r2 = r2_score(y_train, y_train_pred)
print("Training R² Score:", r2)
y_test_pred = model.predict(X_val)

# Calculate R² Score
r2_test = r2_score(y_val, y_test_pred)
print("Test R² Score:", r2_test)